# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [ ]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(26):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-1b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 26 are present

In [ ]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-1b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-1b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-1b"  # ← set this
DESCRIBED_LAYERS = {7, 13, 17, 22}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [9]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [ ]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

### top ten from only described layers

In [ ]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (5, 9, 12, 15)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

### top ten from every other layer

In [ ]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 5, 9, 12, 15)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

### summary

In [ ]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

# get clerp description for described layers (top ten only)

In [ ]:
import requests
import time

def fetch_clerps_batch(features_list, model_id="gemma-3-1b", slug_suffix="gemmascope-2-res-16k", delay=0.3):
    """Fetch clerp descriptions for a list of (layer, feature_idx) tuples."""
    clerps = {}
    
    for i, (layer, feat) in enumerate(features_list):
        source = f"{layer}-{slug_suffix}"
        url = f"https://www.neuronpedia.org/api/feature/{model_id}/{source}/{feat}"
        
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                exps = r.json().get("explanations", [])
                desc = exps[0].get("description") if exps else None
                clerps[(layer, feat)] = desc
                status = "✓" if desc else "✗ (no desc)"
            else:
                clerps[(layer, feat)] = None
                status = f"✗ (HTTP {r.status_code})"
        except Exception as e:
            clerps[(layer, feat)] = None
            status = f"✗ ({str(e)[:30]})"
        
        print(f"  [{i+1}/{len(features_list)}] L{layer:2d} F{feat:5d}: {status}")
        time.sleep(delay)
    
    return clerps

# Extract the top 10 described features
features_to_fetch = [lf for lf, _ in top_10_described_sorted]

# Fetch clerps
print("Fetching clerps for top 10 described features...")
clerps = fetch_clerps_batch(features_to_fetch)

# Print results
print("\n" + "="*70)
print("TOP 10 DESCRIBED FEATURES WITH CLERPS")
print("="*70)
for (layer, feat), score in top_10_described_sorted:
    desc = clerps.get((layer, feat))
    print(f"L{layer:2d} F{feat:5d} (influence={score:8.4f}): {desc or '[no description found]'}")

# intervention

### all 20 features suppressed

In [15]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [ ]:
all_features = top_10_described + top_10_undescribed

intervention_tuples = [
    (layer, TARGET_POS, feat, 0.0)
    for layer, feat in all_features
]

print("=" * 70)
print("INTERVENTION TUPLES (all 20 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")

In [ ]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 20 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)

### suppressed one at a time (top five)

In [ ]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in all_features[:5]:  # just first 5 for speed
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp if available
        clerp_desc = clerps.get((layer, feat)) if (layer, feat) in clerps else None
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")